# 04 - Tabelas gerenciadas e externas

Comparamos o efeito de DROP TABLE em duas cópias das categorias.

| Tipo | Criação neste laboratório | Efeito do DROP TABLE |
|---|---|---|
| Gerenciada | Sem LOCATION na criação da tabela | Remove o registro do catálogo e os arquivos |
| Externa | Com LOCATION apontando para os arquivos | Remove o registro do catálogo e preserva os arquivos |

As duas tabelas usam Delta Lake e ficam no MinIO.
O tipo depende de como a tabela é registrada no catálogo,
não de estar em armazenamento local ou remoto.

In [1]:
import os
from pathlib import Path
import boto3
from botocore.config import Config
from botocore.exceptions import ClientError
from dotenv import load_dotenv
from pyspark.sql import SparkSession
from delta.tables import DeltaTable

raiz = Path.cwd()
if not (raiz / "pyproject.toml").is_file():
    raiz = raiz.parent

load_dotenv(raiz / ".env", override=True)

s3 = boto3.client(
    "s3",
    endpoint_url=os.environ["MINIO_ENDPOINT"],
    aws_access_key_id=os.environ["MINIO_ROOT_USER"],
    aws_secret_access_key=os.environ["MINIO_ROOT_PASSWORD"],
    region_name="us-east-1",
    config=Config(
        signature_version="s3v4",
        s3={"addressing_style": "path"}
    )
)

spark = (
    SparkSession.builder
    .appName("Loja - Tabelas gerenciadas e externas")
    .master("local[2]")
    .config(
        "spark.jars.packages",
        "io.delta:delta-spark_2.12:3.2.0,"
        "org.apache.hadoop:hadoop-aws:3.3.4"
    )
    .config(
        "spark.sql.extensions",
        "io.delta.sql.DeltaSparkSessionExtension"
    )
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog"
    )
    .config("spark.hadoop.fs.s3a.endpoint", os.environ["MINIO_ENDPOINT"])
    .config("spark.hadoop.fs.s3a.access.key", os.environ["MINIO_ROOT_USER"])
    .config("spark.hadoop.fs.s3a.secret.key", os.environ["MINIO_ROOT_PASSWORD"])
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config(
        "spark.hadoop.fs.s3a.impl",
        "org.apache.hadoop.fs.s3a.S3AFileSystem"
    )
    .config("spark.sql.shuffle.partitions", "2")
    .config("spark.databricks.delta.snapshotPartitions", "2")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Versão do Spark:", spark.version)

26/10/13 19:36:50 WARN Utils: Your hostname, YasminNitroV15 resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/10/13 19:36:50 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Ivy Default Cache set to: /home/bfontanayasmin/.ivy2/cache
The jars for the packages stored in: /home/bfontanayasmin/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-bd49ef7b-2274-485b-a5c5-4c7766fac00f;1.0
	confs: [default]


:: loading settings :: url = jar:file:/home/bfontanayasmin/projetos/trabalhoBancodeDados2/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 185ms :: artifacts dl 9ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	io.delta#delta-spark_2.12;3.2.0 from central in [default]
	io.delta#delta-storage;3.2.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.4 from central in [default]
	org.wildfly.openssl#wildfly-openssl;1.0.7.Final from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	--

26/10/13 19:36:50 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


Versão do Spark: 3.5.3


## 1. Preparar o laboratório

Removemos o namespace e os arquivos de uma demonstração anterior
somente na área bronze/laboratorio.

O namespace recebe uma pasta base para suas tabelas gerenciadas.
As cinco tabelas da loja ficam em outros caminhos.

In [2]:
spark.sql("DROP DATABASE IF EXISTS laboratorio_tabelas CASCADE")

PathHadoop = spark._jvm.org.apache.hadoop.fs.Path
pasta_lab = PathHadoop("s3a://bronze/laboratorio")
sistema_arquivos = pasta_lab.getFileSystem(
    spark._jsc.hadoopConfiguration()
)

if sistema_arquivos.exists(pasta_lab):
    sistema_arquivos.delete(pasta_lab, True)

spark.sql("""
    CREATE DATABASE laboratorio_tabelas
    LOCATION 's3a://bronze/laboratorio/gerenciadas'
""")
print("Área de laboratório preparada.")

26/10/13 19:36:54 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


Área de laboratório preparada.


## 2. Criar as duas tabelas

A tabela gerenciada é criada sem LOCATION. O Spark escolhe seu
caminho dentro da pasta base do namespace.

Para a externa, gravamos uma cópia em um caminho específico e
registramos esse caminho usando LOCATION.

In [3]:
spark.sql("""
    CREATE TABLE laboratorio_tabelas.gerenciada
    USING DELTA
    AS SELECT * FROM delta.`s3a://bronze/categorias`
""")

caminho_externa = "s3a://bronze/laboratorio/externas/categorias"

(
    spark.read.format("delta")
    .load("s3a://bronze/categorias")
    .write.format("delta")
    .mode("overwrite")
    .save(caminho_externa)
)

spark.sql(f"""
    CREATE TABLE laboratorio_tabelas.externa
    USING DELTA
    LOCATION '{caminho_externa}'
""")

caminho_gerenciada = spark.sql(
    "DESCRIBE DETAIL laboratorio_tabelas.gerenciada"
).select("location").first()[0]

for nome in ["gerenciada", "externa"]:
    tabela = f"laboratorio_tabelas.{nome}"
    tipo = spark.catalog.getTable(tabela).tableType
    quantidade = spark.table(tabela).count()
    print(f"{nome}: tipo={tipo}, registros={quantidade}")
    assert quantidade == 2

assert spark.catalog.getTable(
    "laboratorio_tabelas.gerenciada"
).tableType == "MANAGED"

assert spark.catalog.getTable(
    "laboratorio_tabelas.externa"
).tableType == "EXTERNAL"

assert sistema_arquivos.exists(PathHadoop(caminho_gerenciada))
assert sistema_arquivos.exists(PathHadoop(caminho_externa))
print("Arquivos das duas tabelas encontrados no MinIO.")

26/10/13 19:36:58 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


gerenciada: tipo=MANAGED, registros=2


externa: tipo=EXTERNAL, registros=2
Arquivos das duas tabelas encontrados no MinIO.


## 3. Remover as tabelas e verificar os arquivos

DROP TABLE remove os dois registros do catálogo.
Verificamos diretamente no armazenamento quais arquivos permanecem.

Na externa, os metadados do Delta em _delta_log também permanecem
junto dos dados. O que é removido é o registro da tabela no catálogo.

In [4]:
spark.sql("DROP TABLE laboratorio_tabelas.gerenciada")
spark.sql("DROP TABLE laboratorio_tabelas.externa")

gerenciada_existe = sistema_arquivos.exists(
    PathHadoop(caminho_gerenciada)
)
externa_existe = sistema_arquivos.exists(
    PathHadoop(caminho_externa)
)

assert not spark.catalog.tableExists("laboratorio_tabelas.gerenciada")
assert not spark.catalog.tableExists("laboratorio_tabelas.externa")
assert not gerenciada_existe
assert externa_existe
assert DeltaTable.isDeltaTable(spark, caminho_externa)

print("Depois do DROP TABLE:")
print("Arquivos da gerenciada existem:", gerenciada_existe)
print("Arquivos da externa existem:", externa_existe)

preservados = spark.read.format("delta").load(caminho_externa)
assert preservados.count() == 2
preservados.orderBy("id_categoria").show(truncate=False)

Depois do DROP TABLE:
Arquivos da gerenciada existem: False
Arquivos da externa existem: True


+------------+--------------+
|id_categoria|nome_categoria|
+------------+--------------+
|1           |Informática   |
|2           |Papelaria     |
+------------+--------------+



## 4. Registrar novamente a externa

Como os dados e o log Delta foram preservados, podemos registrar
novamente a tabela apontando para o mesmo caminho.

No trabalho 1, as tabelas Delta usavam LOCATION no disco local.
No trabalho 2, as cinco tabelas do bronze também usam LOCATION,
agora no MinIO. Nos dois casos, são externas.

O laboratório adicional demonstra uma tabela gerenciada e comprova
a diferença no comportamento do DROP TABLE.

Criar um DataFrame, por si só, não determina o tipo da tabela:
a maneira de gravar e registrar os dados também importa.

In [5]:
spark.sql(f"""
    CREATE TABLE laboratorio_tabelas.externa
    USING DELTA
    LOCATION '{caminho_externa}'
""")

assert spark.table("laboratorio_tabelas.externa").count() == 2
assert spark.read.format("delta").load(
    "s3a://bronze/categorias"
).count() == 2

print("Gerenciada: DROP TABLE removeu o registro e os arquivos.")
print("Externa: DROP TABLE removeu o registro e preservou os arquivos.")
print("Externa registrada novamente: dois registros recuperados.")
print("Categorias da loja preservadas: dois registros.")
print("Comparação concluída e validada.")

spark.stop()
print("Sessão Spark encerrada.")

Gerenciada: DROP TABLE removeu o registro e os arquivos.
Externa: DROP TABLE removeu o registro e preservou os arquivos.
Externa registrada novamente: dois registros recuperados.
Categorias da loja preservadas: dois registros.
Comparação concluída e validada.


Sessão Spark encerrada.
